# Module 3 — Factor Orthogonalization

## Overview

M2 identified **28 of 41 factors** as individually priced in the univariate Fama MacBeth
cross-section. However, many of these factors are highly correlated, for instance, 100 pairs exceed $|r| > 0.70$ and 10 pairs exceed $|r| > 0.95$.
A factor can appear priced in M2 simply because it is correlated
with a genuinely priced factor, not because it carries independent
information.

M3 asks the harder question: **which factors add independent
pricing information beyond what all other factors already capture?**

## The Spanning Test

For each factor $j$, we run a time-series regression of its returns
on the returns of all other 40 factors:

$$F_{j,t} = \alpha_j + \sum_{k \neq j} \beta_{jk} F_{k,t}
  + \varepsilon_{j,t}$$

- If $\alpha_j = 0$: factor $j$ is **spanned** — its returns can be
  fully replicated by a portfolio of the other 40 factors. It adds
  no independent information and is redundant.
- If $\alpha_j \neq 0$: factor $j$ is **non-spanned** — it has a
  component that cannot be replicated by any combination of the
  other factors. That component is its independent contribution.

We test $H_0: \alpha_j = 0$ using a $t$-statistic with
Newey-West HAC standard errors ($L = \lfloor T^{1/4} \rfloor$
lags), the same correction used in M2.

This is the standard test used in Fama \& French (2018) to evaluate
whether new factors add to an existing model, and in Barillas \&
Shanken (2017) to compare competing factor models.

## Orthogonalized Factor Returns

For each factor $j$, we construct its **orthogonalized return** —
the component not explained by the other 40 factors:

$$\tilde{F}_{j,t} = \hat{\alpha}_j + \hat{\varepsilon}_{j,t}
  = F_{j,t} - \sum_{k \neq j} \hat{\beta}_{jk} F_{k,t}$$

By construction, $\tilde{F}_{j,t}$ is uncorrelated with every other
factor. It represents the pure independent signal of factor $j$.
The Sharpe ratio of $\tilde{F}_{j,t}$ measures the risk-adjusted
return that is genuinely unique to factor $j$ and cannot be
captured by any combination of the other 40 factors.

## Connection to M2

| Module | Question | Method |
|--------|----------|--------|
| M2 Univariate FM | Does factor $j$ predict returns on its own? | Cross-sectional |
| M3 Spanning | Does factor $j$ add anything beyond all others? | Time-series |

A factor is most credible if it passes **both** tests:
priced in M2 ($|t_{\text{NW}}| > 1.96$) and non-spanned in M3
($|\alpha_j| > 0$ significantly). Factors that pass M2 but fail M3
are redundant proxies. These are the factors we drop before M5
portfolio construction.

## Inputs and Outputs

**Input:** `outputs/M1/factor_returns_41.parquet`

**Outputs** (saved to `outputs/M3/`):
- `spanning_results.csv` -- $\alpha_j$, $t$-stat, $R^2$ per factor
- `factor_returns_orthogonal.parquet` -- $\tilde{F}_{j,t}$ for all 41 factors
- `orthogonal_stats.csv` -- Sharpe, mean, std of orthogonalized factors
- `figures/` -- alpha bar chart, spanning $R^2$ heatmap

In [1]:
# Imports and configuration

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import os, warnings
warnings.filterwarnings('ignore')

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook"

# ── Paths ─────────────────────────────────────────────────────
PROJECT_ROOT = (r"C:\Users\amosa\Documents\My Graduate School"
                r"\SPRING 2026\Courses\AMS520_Machine Learning in Finance"
                r"\Project\General ML4Trading Resource"
                r"\Personal_Fundamental Factor Investing"
                r"\fundamental-factor-investing")
M1_OUT  = os.path.join(PROJECT_ROOT, "outputs", "M1")
M2_OUT  = os.path.join(PROJECT_ROOT, "outputs", "M2")
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "M3")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── 41 characteristics by economic family ─────────────────────
CHAR_FAMILIES = {
    "Value":         ["BEME","E2P","CF2P","D2P","S2P","A2ME"],
    "Profitability": ["PROF","ROE","ROA","OP","PM","PCM","RNA"],
    "Investment":    ["Investment","NOA","DPI2A","NI","OA","AC"],
    "Momentum":      ["r12_2","r2_1","r12_7","r36_13",
                      "LT_Rev","SUV","Rel2High"],
    "Risk":          ["Beta","IdioVol","Variance",
                      "Spread","LTurnover","LME"],
    "Other":         ["Q","C","AT","ATO","CTO",
                      "D2A","FC2Y","Lev","SGA2S"],
}
ALL_41 = [c for fam in CHAR_FAMILIES.values() for c in fam]

FAMILY_COLORS = {
    "Value":         "#1f77b4",
    "Profitability": "#2ca02c",
    "Investment":    "#ff7f0e",
    "Momentum":      "#d62728",
    "Risk":          "#9467bd",
    "Other":         "#8c564b",
}

# ── Priced factors from M2 (univariate, |t_NW| > 1.96) ────────
M2_PRICED = [
    "BEME","E2P","CF2P","D2P","S2P","A2ME",          # Value: 6/6
    "PROF",                                            # Profitability: 1/7
    "Investment","NOA","DPI2A","NI","OA","AC",         # Investment: 6/6
    "r2_1","r36_13","LT_Rev","SUV","Rel2High",         # Momentum: 5/7
    "IdioVol","Variance","Spread","LTurnover","LME",   # Risk: 5/6
    "Q","AT","CTO","FC2Y","Lev",                       # Other: 5/9
]

print(f"Output directory: {OUT_DIR}")
print(f"Total factors:    {len(ALL_41)}")
print(f"M2 priced:        {len(M2_PRICED)}/41")

Output directory: C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS520_Machine Learning in Finance\Project\General ML4Trading Resource\Personal_Fundamental Factor Investing\fundamental-factor-investing\outputs\M3
Total factors:    41
M2 priced:        28/41


In [2]:
# Loading factor returns data from Module 1

print("Loading factor returns from M1...")

factor_returns = pd.read_parquet(
    os.path.join(M1_OUT, "factor_returns_41.parquet"))
factor_returns.index = (pd.to_datetime(factor_returns.index)
                        + pd.offsets.MonthEnd(0))
factor_returns = factor_returns[ALL_41].copy()

# Load M2 univariate results for reference
fm_uni = pd.read_csv(
    os.path.join(M2_OUT, "lambda_univariate.csv"),
    index_col=0)

T = len(factor_returns)
L = int(np.floor(T ** 0.25))

print(f"Factor returns: {factor_returns.shape}")
print(f"Date range:     {factor_returns.index.min().date()} — "
      f"{factor_returns.index.max().date()}")
print(f"T = {T},  Newey-West lag L = {L}")

# Quick null check
null_counts = factor_returns.isnull().sum()
factors_with_nulls = null_counts[null_counts > 0]
if len(factors_with_nulls):
    print(f"\nFactors with nulls:")
    for char, n in factors_with_nulls.items():
        print(f"  {char}: {n} nulls ({n/T:.1%})")
else:
    print("\nNo nulls in factor returns ✓")

Loading factor returns from M1...
Factor returns: (696, 41)
Date range:     1967-01-31 — 2024-12-31
T = 696,  Newey-West lag L = 5

Factors with nulls:
  Spread: 326 nulls (46.8%)


---

## Spanning Regressions

For each factor $j$, we estimate:

$$F_{j,t} = \alpha_j + \boldsymbol{\beta}_j^\top
  \mathbf{F}_{-j,t} + \varepsilon_{j,t}$$

where $\mathbf{F}_{-j,t}$ is the vector of all other 40 factor
returns in month $t$.

**Interpretation of $R^2$:** A high $R^2$ means factor $j$'s
returns are well-explained by the other 40 factors - it is
nearly redundant. A low $R^2$ means factor $j$ moves
independently - it brings genuinely new information.

**Interpretation of $\alpha_j$:** The intercept is the average
monthly return of factor $j$ that cannot be attributed to its
co-movement with other factors. If $\alpha_j > 0$ and significant,
factor $j$ earns an independent risk premium above what the other
factors already price.

**Handling missing data:** Spread has pre-1986 NaN values.
For each spanning regression we use the intersection of
non-missing months for all factors involved. This means Spread's
spanning regression uses the post-1986 subsample only.

In [3]:
# Spanning regressions

# ── Newey-West HAC standard error ─────────────────────────────
def newey_west_se(series, lags):
    x  = series.dropna().values
    T_ = len(x)
    if T_ < lags + 2:
        return np.nan
    dev    = x - x.mean()
    nw_var = (dev ** 2).mean()
    for ell in range(1, lags + 1):
        w       = 1.0 - ell / (lags + 1)
        nw_var += 2 * w * (dev[ell:] * dev[:-ell]).mean()
    return np.sqrt(max(nw_var, 0.0) / T_)


print("Running spanning regressions...")
print(f"41 regressions, each using 40 factors as regressors\n")

spanning_results = []
beta_matrix      = pd.DataFrame(index=ALL_41, columns=ALL_41,
                                 dtype=float)
orthogonal_rets  = {}

for j, char in enumerate(ALL_41):
    # Use months where both the target factor and all regressors
    # are non-null
    other_chars = [c for c in ALL_41 if c != char]

    # Find complete cases
    cols_needed = [char] + other_chars
    complete    = factor_returns[cols_needed].dropna()

    if len(complete) < 60:
        print(f"  WARNING: {char} has only {len(complete)} "
              f"complete months — skipping")
        continue

    y = complete[char].values
    X = np.column_stack([
        np.ones(len(complete)),
        complete[other_chars].values
    ])

    # OLS spanning regression
    try:
        coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    except np.linalg.LinAlgError:
        print(f"  WARNING: singular matrix for {char} — skipping")
        continue

    alpha   = coef[0]
    betas   = coef[1:]
    y_hat   = X @ coef
    resid   = y - y_hat
    ss_res  = resid @ resid
    ss_tot  = ((y - y.mean()) ** 2).sum()
    r2      = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    adj_r2  = (1 - (1 - r2) * (len(y) - 1)
               / (len(y) - X.shape[1])) if r2 is not np.nan else np.nan

    # Newey-West SE on alpha
    # Alpha SE from HAC covariance of residuals
    resid_s  = pd.Series(resid, index=complete.index)
    alpha_se = newey_west_se(resid_s + alpha, L)
    t_alpha  = alpha / alpha_se if (alpha_se and alpha_se > 0) else np.nan

    # Information ratio of alpha
    ir = alpha / resid.std() * np.sqrt(12) if resid.std() > 0 else np.nan

    # Store betas
    for k, other in enumerate(other_chars):
        beta_matrix.loc[char, other] = betas[k]

    # Orthogonalized return series = alpha + residual
    orth_ret = pd.Series(alpha + resid, index=complete.index)
    orthogonal_rets[char] = orth_ret

    # Family
    family = next(f for f, cs in CHAR_FAMILIES.items() if char in cs)

    # Sharpe of original vs orthogonalized
    sharpe_orig = (y.mean() / y.std() * np.sqrt(12)
                   if y.std() > 0 else np.nan)
    sharpe_orth = (orth_ret.mean() / orth_ret.std() * np.sqrt(12)
                   if orth_ret.std() > 0 else np.nan)

    spanning_results.append({
        'Characteristic':    char,
        'Family':            family,
        'Alpha (ann %)':     round(alpha * 12 * 100, 3),
        'Alpha SE (NW)':     round(alpha_se, 6) if alpha_se else np.nan,
        't (alpha)':         round(t_alpha, 3) if not np.isnan(t_alpha) else np.nan,
        'R²':                round(r2, 4),
        'Adj R²':            round(adj_r2, 4),
        'Sharpe (original)': round(sharpe_orig, 3),
        'Sharpe (orthog)':   round(sharpe_orth, 3),
        'IR (ann)':          round(ir, 3),
        'N months':          len(complete),
        'Spanned':           abs(t_alpha) < 1.96 if not np.isnan(t_alpha) else None,
    })

    if (j + 1) % 10 == 0:
        print(f"  Completed {j+1}/41 factors...")

spanning = pd.DataFrame(spanning_results).set_index('Characteristic')

# Assemble orthogonalized returns into DataFrame
orth_returns = pd.DataFrame(orthogonal_rets)
orth_returns.index = pd.to_datetime(orth_returns.index)

print(f"\nSpanning regressions complete.")
print(f"Orthogonalized returns shape: {orth_returns.shape}")

n_spanned    = spanning['Spanned'].sum()
n_non_spanned = (~spanning['Spanned']).sum()
print(f"\nSpanned (α not significant):     {n_spanned}/41")
print(f"Non-spanned (α significant):     {n_non_spanned}/41")

Running spanning regressions...
41 regressions, each using 40 factors as regressors

  Completed 10/41 factors...
  Completed 20/41 factors...
  Completed 30/41 factors...
  Completed 40/41 factors...

Spanning regressions complete.
Orthogonalized returns shape: (370, 41)

Spanned (α not significant):     14/41
Non-spanned (α significant):     27/41


---

## Results: Spanning Test

The table below reports for each factor:

- **Alpha (ann %):** annualized monthly alpha from the spanning
  regression - the independent return component
- **t(alpha):** Newey-West $t$-statistic testing $H_0: \alpha_j = 0$
- **R²:** fraction of factor $j$'s variance explained by the
  other 40 factors. High R² = nearly redundant.
- **Sharpe (original):** Sharpe ratio of the raw factor return
- **Sharpe (orthog):** Sharpe ratio of the orthogonalized return
  $\tilde{F}_{j,t}$ - the pure independent signal

A factor is **non-spanned** if $|t(\alpha)| > 1.96$.
Non-spanned factors carry genuinely independent pricing information.
Spanned factors are redundant proxies for other priced factors.

The comparison of Sharpe (original) vs Sharpe (orthog) shows how
much of each factor's return performance survives after removing
the component shared with other factors.

In [4]:
# Results

print(f"{'─'*88}")
print(f"{'Characteristic':<16} {'Family':<15} "
      f"{'Alpha%':>8} {'t(α)':>7} {'R²':>6} "
      f"{'SR(orig)':>9} {'SR(orth)':>9}  Status")
print(f"{'─'*88}")

n_non_spanned_priced = 0
n_spanned_priced     = 0

for fam in CHAR_FAMILIES:
    print(f"\n  ── {fam} ──")
    for char in CHAR_FAMILIES[fam]:
        if char not in spanning.index:
            continue
        row    = spanning.loc[char]
        ta     = row['t (alpha)']
        spanned = row['Spanned']
        m2_priced = char in M2_PRICED

        status = ""
        if not spanned and m2_priced:
            status = "✓ priced + non-spanned"
            n_non_spanned_priced += 1
        elif not spanned and not m2_priced:
            status = "△ non-spanned, not M2 priced"
        elif spanned and m2_priced:
            status = "— spanned (redundant)"
            n_spanned_priced += 1
        else:
            status = "  insignificant"

        sig = ("**" if abs(ta) > 3.00
               else "*" if abs(ta) > 1.96
               else "  ")

        print(f"  {char:<16} {row['Family']:<15} "
              f"{row['Alpha (ann %)']:>8.2f} "
              f"{ta:>6.2f}{sig} "
              f"{row['R²']:>6.3f} "
              f"{row['Sharpe (original)']:>9.3f} "
              f"{row['Sharpe (orthog)']:>9.3f}  "
              f"{status}")

print(f"\n{'─'*88}")
print(f"* |t|>1.96   ** |t|>3.00")
print(f"\nSummary:")
print(f"  Non-spanned AND M2 priced: {n_non_spanned_priced}/41  "
      f"← strongest candidates for M5")
print(f"  Spanned but M2 priced:     {n_spanned_priced}/41  "
      f"← redundant proxies")
print(f"  Non-spanned, not M2 priced:{spanning['Spanned'].eq(False).sum() - n_non_spanned_priced}/41  "
      f"← theoretically interesting, weak signal")

────────────────────────────────────────────────────────────────────────────────────────
Characteristic   Family            Alpha%    t(α)     R²  SR(orig)  SR(orth)  Status
────────────────────────────────────────────────────────────────────────────────────────

  ── Value ──
  BEME             Value               2.48   5.86**  0.989     3.606     1.295  ✓ priced + non-spanned
  E2P              Value               8.02  15.70**  0.974     0.753     2.971  ✓ priced + non-spanned
  CF2P             Value              -5.48 -10.22**  0.982     1.240    -2.128  ✓ priced + non-spanned
  D2P              Value              -0.02  -0.04    0.950     0.909    -0.006  — spanned (redundant)
  S2P              Value              -6.12 -14.88**  0.986     2.293    -2.770  ✓ priced + non-spanned
  A2ME             Value               1.47   4.85**  0.991     3.087     0.791  ✓ priced + non-spanned

  ── Profitability ──
  PROF             Profitability      -1.75  -3.44**  0.925     0.823    -0.

---

## Reading the M3 Results

- **20 factors are priced (M2) AND non-spanned (M3).**

  These are the strongest candidates for M5 portfolio construction. They earn independent risk premia that cannot be replicated by any combination of the other 40 factors.

- **8 factors are priced in M2 but spanned in M3 - the redundant proxies:**

  D2P, Investment, DPI2A, LT\_Rev, IdioVol, AT, CTO, Lev. These factors appeared significant in M2 only because they are correlated with other priced factors. Their independent alpha is statistically zero. They add nothing beyond what other factors already capture.

- **7 factors are non-spanned but not M2 priced:**

  ROE, PM, RNA, r12\_2, r12\_7, Beta, C. The most interesting case here is r12\_2 as it has a massive alpha of −19.14% (t = −15.93) in the spanning regression, meaning after controlling for all other momentum factors it has strong independent content, yet its univariate risk price in M2 was essentially zero (t = 0.07). This is a genuine puzzle: r12\_2 has independent variation but that variation is not priced. Beta (alpha = −5.78%, t = −4.36) is similar -- it has independent content but does not earn a risk premium.

**Three findings worth highlighting specifically:**

- **Rel2High** has the highest orthogonalized Sharpe at 8.652 - even after removing everything it shares with the other 40 factors, its independent component earns an extraordinary risk-adjusted return. This is the single most irreplaceable factor in the dataset.

- **The value cluster** (BEME, E2P, CF2P, S2P, A2ME all non-spanned) is surprising - despite correlations exceeding 0.97, each value factor retains a significant independent alpha. This means each is capturing a slightly different dimension of cheapness that the others miss.

- **The profitability cluster** (ROA, OP, PCM all spanned with R² > 0.86) confirms what M1 and M2 suggested: most profitability factors are not independently priced. Only PROF and ROE have independent content, and only PROF was priced in M2.

In [8]:
# Spanning R^2 barchart

print("Generating spanning R² chart...")

# Sort by R² descending — most redundant factors first
spanning_sorted = spanning.sort_values('R²', ascending=False)

colors = [FAMILY_COLORS[spanning.loc[c, 'Family']]
          for c in spanning_sorted.index]
alphas_sig = [
    0.9 if not spanning.loc[c, 'Spanned'] else 0.4
    for c in spanning_sorted.index
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x           = spanning_sorted.index.tolist(),
    y           = spanning_sorted['R²'].values * 100,
    marker_color= colors,
    marker_opacity = alphas_sig,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "R²: %{y:.1f}%<br>"
        "<extra></extra>"
    )
))

# Reference line at 50%
fig.add_hline(y=50,
              line=dict(color='red', width=1.5, dash='dash'),
              annotation_text="50% — high redundancy",
              annotation_position="right")

fig.update_layout(
    title=dict(
        text=("<b>Spanning Regression R² by Factor</b><br>"
              "<sup>Fraction of each factor's variance explained by "
              "the other 40 factors | "
              "Opaque = non-spanned (α significant) | "
              "Faded = spanned (redundant)</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Characteristic',
               tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(title='Spanning R² (%)', ticksuffix='%',
               showgrid=True, gridcolor='rgba(200,200,200,0.3)',
               range=[0, 105]),
    width=1200, height=500,
    margin=dict(l=60, r=160, t=100, b=130),
)
fig.show()
fig.write_image(os.path.join(FIG_DIR, "spanning_r2.png"), scale=2)
print("Saved: spanning_r2.png")

Generating spanning R² chart...


Saved: spanning_r2.png


In [9]:
# Alpha bar chart

print("Generating alpha bar chart...")

# Sort by alpha magnitude
spanning_sorted_alpha = spanning.sort_values('Alpha (ann %)',
                                              ascending=False)

colors_a = [FAMILY_COLORS[spanning.loc[c, 'Family']]
            for c in spanning_sorted_alpha.index]
opacities = [
    0.9 if not spanning.loc[c, 'Spanned'] else 0.35
    for c in spanning_sorted_alpha.index
]

errors = (spanning_sorted_alpha['Alpha SE (NW)']
          * np.sqrt(12) * 100 * 1.96).values

fig = go.Figure()

fig.add_trace(go.Bar(
    x              = spanning_sorted_alpha.index.tolist(),
    y              = spanning_sorted_alpha['Alpha (ann %)'].values,
    error_y        = dict(type='data', array=errors,
                          visible=True, thickness=1.5, width=4),
    marker_color   = colors_a,
    marker_opacity = opacities,
    hovertemplate  = (
        "<b>%{x}</b><br>"
        "Alpha (ann%): %{y:.2f}%<br>"
        "<extra></extra>"
    )
))

fig.add_hline(y=0,
              line=dict(color='black', width=1, dash='dash'),
              opacity=0.5)

fig.update_layout(
    title=dict(
        text=("<b>Spanning Regression Alpha by Factor</b><br>"
              "<sup>Annualized independent return component | "
              "Error bars = NW 95% CI | "
              "Opaque = non-spanned | Faded = spanned</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Characteristic',
               tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(title='Alpha (ann %)', ticksuffix='%',
               showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    width=1100, height=500,
    margin=dict(l=60, r=60, t=100, b=130),
)
fig.show()
fig.write_image(os.path.join(FIG_DIR, "spanning_alpha.png"), scale=2)
print("Saved: spanning_alpha.png")

Generating alpha bar chart...


Saved: spanning_alpha.png


----

## Orthogonalized Factor Returns

Each factor's orthogonalized return $\tilde{F}_{j,t}$ is the
residual from its spanning regression plus the alpha:

$$\tilde{F}_{j,t} = F_{j,t} - \sum_{k \neq j} \hat{\beta}_{jk} F_{k,t} = \hat{\alpha}_j + \hat{\varepsilon}_{j,t}$$

This is the **residual maker** $M = I - X(X'X)^{-1}X'$ applied to each factor. The orthogonalized factor $\tilde{F}_{j,t}$ is by construction uncorrelated with all other factors. It represents the pure, independent signal of factor $j$.

By construction:
- $\text{Corr}(\tilde{F}_{j,t}, F_{k,t}) = 0$ for all $k \neq j$
- $\mathbb{E}[\tilde{F}_{j,t}] = \alpha_j$

The Sharpe ratio of $\tilde{F}_{j,t}$ measures the standalone
risk-adjusted return of factor $j$'s independent component.
A high orthogonalized Sharpe means the factor earns strong
returns that cannot be replicated by any combination of the
other 40 factors - this is the most rigorous measure of a
factor's independent value.

The comparison **Sharpe (original) vs Sharpe (orthog)** tells us
how much of each factor's apparent performance is due to its own
signal versus its co-movement with other factors. A large drop
from original to orthogonalized Sharpe indicates the factor's
returns are mostly attributable to other factors - a sign of
redundancy even if the factor appeared priced in M2.

In [10]:
# Orthogonalized Sharpe comparison plot

print("Generating orthogonalized Sharpe comparison...")

# Factors sorted by orthogonalized Sharpe, descending
spanning_sr = spanning.sort_values('Sharpe (orthog)', ascending=False)

chars  = spanning_sr.index.tolist()
sr_orig = spanning_sr['Sharpe (original)'].values
sr_orth = spanning_sr['Sharpe (orthog)'].values
cols    = [FAMILY_COLORS[spanning_sr.loc[c,'Family']] for c in chars]

fig = go.Figure()

fig.add_trace(go.Bar(
    name         = 'Original Sharpe',
    x            = chars,
    y            = sr_orig,
    marker_color = cols,
    opacity      = 0.4,
    hovertemplate="<b>%{x}</b><br>Original SR: %{y:.3f}<extra></extra>"
))

fig.add_trace(go.Bar(
    name         = 'Orthogonalized Sharpe',
    x            = chars,
    y            = sr_orth,
    marker_color = cols,
    opacity      = 0.9,
    hovertemplate="<b>%{x}</b><br>Orthog SR: %{y:.3f}<extra></extra>"
))

fig.add_hline(y=0,
              line=dict(color='black', width=1, dash='dash'),
              opacity=0.5)

fig.update_layout(
    title=dict(
        text=("<b>Original vs Orthogonalized Sharpe Ratio</b><br>"
              "<sup>Sorted by orthogonalized Sharpe (descending) | "
              "Drop from original to orthogonalized = redundancy</sup>"),
        font=dict(size=14), x=0.0),
    xaxis=dict(title='Characteristic',
               tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(title='Annualized Sharpe Ratio',
               showgrid=True, gridcolor='rgba(200,200,200,0.3)'),
    barmode='overlay',
    legend=dict(orientation='h', yanchor='bottom',
                y=1.02, xanchor='right', x=1),
    width=1100, height=500,
    margin=dict(l=60, r=60, t=100, b=130),
)
fig.show()
fig.write_image(os.path.join(FIG_DIR, "sharpe_comparison.png"), scale=2)
print("Saved: sharpe_comparison.png")

Generating orthogonalized Sharpe comparison...


Saved: sharpe_comparison.png


In [11]:
print("=== Saving M3 outputs ===\n")

# Spanning results
spanning.to_csv(os.path.join(OUT_DIR, "spanning_results.csv"))
print("Saved: spanning_results.csv")

# Orthogonalized factor returns
orth_returns.to_parquet(
    os.path.join(OUT_DIR, "factor_returns_orthogonal.parquet"))
print("Saved: factor_returns_orthogonal.parquet")

# Orthogonalized factor statistics
orth_stats_list = []
for char in ALL_41:
    if char not in orth_returns.columns:
        continue
    r  = orth_returns[char].dropna()
    orth_stats_list.append({
        'Characteristic': char,
        'Family':         next(f for f, cs in CHAR_FAMILIES.items()
                               if char in cs),
        'Mean (ann %)':   round(r.mean() * 12 * 100, 3),
        'Std (ann %)':    round(r.std()  * np.sqrt(12) * 100, 3),
        'Sharpe':         round(r.mean() * 12
                                / (r.std() * np.sqrt(12)), 3)
                          if r.std() > 0 else np.nan,
        'N months':       len(r),
    })

orth_stats = pd.DataFrame(orth_stats_list).set_index('Characteristic')
orth_stats.to_csv(os.path.join(OUT_DIR, "orthogonal_stats.csv"))
print("Saved: orthogonal_stats.csv")

# File inventory
print(f"\nAll M3 output files:")
for f in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, f)
    if os.path.isfile(full):
        print(f"  {f:<48} {os.path.getsize(full)/1024:>7.1f} KB")

# ── Final summary ──────────────────────────────────────────────
n_non_spanned = (~spanning['Spanned']).sum()
n_spanned     = spanning['Spanned'].sum()
n_both        = sum(1 for c in spanning.index
                    if not spanning.loc[c,'Spanned']
                    and c in M2_PRICED)
n_redundant   = sum(1 for c in spanning.index
                    if spanning.loc[c,'Spanned']
                    and c in M2_PRICED)

top5_orth = (orth_stats['Sharpe']
             .nlargest(5)
             .reset_index()
             .to_string(index=False))

print(f"""
{'='*60}
M3 COMPLETE
{'='*60}
Factors:              41
Spanning regressions: 41 (each vs 40 others)
Newey-West lag:       L = {L}

Spanning results:
  Non-spanned (α significant):    {n_non_spanned}/41
  Spanned (redundant):            {n_spanned}/41

Cross-referencing with M2:
  Priced (M2) AND non-spanned:    {n_both}/41  ← strongest
  Priced (M2) but spanned:        {n_redundant}/41  ← redundant

Top 5 factors by orthogonalized Sharpe:
{top5_orth}

Feeds into:  M4 — Signal Evaluation
             M5 — Portfolio Construction
{'='*60}
""")

=== Saving M3 outputs ===

Saved: spanning_results.csv
Saved: factor_returns_orthogonal.parquet
Saved: orthogonal_stats.csv

All M3 output files:
  factor_returns_orthogonal.parquet                  161.9 KB
  orthogonal_stats.csv                                 1.6 KB
  spanning_results.csv                                 3.3 KB

M3 COMPLETE
Factors:              41
Spanning regressions: 41 (each vs 40 others)
Newey-West lag:       L = 5

Spanning results:
  Non-spanned (α significant):    27/41
  Spanned (redundant):            14/41

Cross-referencing with M2:
  Priced (M2) AND non-spanned:    20/41  ← strongest
  Priced (M2) but spanned:        8/41  ← redundant

Top 5 factors by orthogonalized Sharpe:
Characteristic  Sharpe
      Rel2High   8.652
           E2P   2.971
          r2_1   2.204
        Spread   1.877
             C   1.556

Feeds into:  M4 — Signal Evaluation
             M5 — Portfolio Construction

